In [ ]:
from pathlib import Path

content = Path("/content")
h5_files = sorted(
    path for path in content.rglob("*")
    if path.is_file() and path.suffix.lower() in {".h5", ".hdf5"}
)

print(f"Found {len(h5_files)} HDF5 files:")
for path in h5_files:
    print(path)

In [ ]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt

# Search Colab's session storage, including subfolders
all_h5_files = sorted(
    path for path in Path("/content").rglob("*")
    if path.is_file()
    and path.name.startswith("VNP46A4")
    and path.suffix.lower() == ".h5"
)

print(f"Found {len(all_h5_files)} VNP46A4 HDF5 files:")
for path in all_h5_files:
    print(path)

def find_year_file(year):
    # NASA filenames typically contain A2012001 or A2023001
    matches = [
        path for path in all_h5_files
        if f"A{year}001" in path.name
    ]

    if not matches:
        raise FileNotFoundError(
            f"No VNP46A4 file for {year} found under /content. "
            "Check the file list above or upload the file into this runtime."
        )

    if len(matches) > 1:
        raise ValueError(
            f"Found multiple files for {year}; choose the matching tile:\n"
            + "\n".join(str(path) for path in matches)
        )

    return matches[0]

years = [2012, 2023]
files = [find_year_file(year) for year in years]

layer = (
    "HDFEOS/GRIDS/VIIRS_Grid_DNB_2d/Data Fields/"
    "AllAngle_Composite_Snow_Free"
)
lat_path = "HDFEOS/GRIDS/VIIRS_Grid_DNB_2d/Data Fields/lat"
lon_path = "HDFEOS/GRIDS/VIIRS_Grid_DNB_2d/Data Fields/lon"

radiance = []

for path in files:
    with h5py.File(path, "r") as f:
        dataset = f[layer]
        data = dataset[()].astype(np.float32)
        fill = float(np.asarray(dataset.attrs.get("_FillValue", -999.9)).squeeze())
        data[data == fill] = np.nan
        radiance.append(data)

with h5py.File(files[0], "r") as f:
    lat = f[lat_path][()]
    lon = f[lon_path][()]

# Use one shared display scale for both maps
vmax = np.nanpercentile(
    np.concatenate([data.ravel() for data in radiance]),
    99.5
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

for ax, data, year in zip(axes, radiance, years):
    im = ax.imshow(
        data,
        origin="upper",
        extent=[lon.min(), lon.max(), lat.min(), lat.max()],
        cmap="magma",
        vmin=0,
        vmax=vmax
    )
    ax.set_title(str(year))
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

fig.colorbar(im, ax=axes, label="Night-light radiance")
plt.show()

In [ ]:
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap

files = {
    2012: next(Path("/content").glob("VNP46A4.A2012*.h5")),
    2023: next(Path("/content").glob("VNP46A4.A2023*.h5")),
}

base = "HDFEOS/GRIDS/VIIRS_Grid_DNB_2d/Data Fields/"
radiance_layer = base + "AllAngle_Composite_Snow_Free"
quality_layer = base + "AllAngle_Composite_Snow_Free_Quality"

arrays = {}

for year, path in files.items():
    with h5py.File(path, "r") as f:
        arrays[year] = {
            "radiance": f[radiance_layer][()],
            "quality": f[quality_layer][()],
        }

r12, q12 = arrays[2012]["radiance"], arrays[2012]["quality"]
r23, q23 = arrays[2023]["radiance"], arrays[2023]["quality"]

# Keep pixels marked good in both years and remove fill values.
valid = (
    (q12 == 0) & (q23 == 0) &
    np.isfinite(r12) & np.isfinite(r23) &
    (r12 > -900) & (r23 > -900)
)

delta = np.full(r12.shape, np.nan, dtype="float32")
delta[valid] = r23[valid] - r12[valid]

print(f"Good pixels in both years: {valid.mean():.1%}")
print(f"Change range in those pixels: {np.nanmin(delta):.2f} to {np.nanmax(delta):.2f}")

with h5py.File(files[2012], "r") as f:
    lat = f[base + "lat"][()]
    lon = f[base + "lon"][()]

limit = np.nanpercentile(np.abs(delta), 98)
change_colors = LinearSegmentedColormap.from_list(
    "change", ["#e98224", "#f3f1ed", "#7139a5"]
)

plt.figure(figsize=(9, 7))
plt.imshow(
    delta,
    origin="upper",
    extent=[lon.min(), lon.max(), lat.min(), lat.max()],
    cmap=change_colors,
    norm=TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)
)
plt.colorbar(label="Radiance change (2023 − 2012)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Dubai–Abu Dhabi night-light radiance change")
plt.show()

In [ ]:
from pathlib import Path
import h5py
import numpy as np

files = sorted(Path("/content").glob("VNP46A4*.h5"))
base = "HDFEOS/GRIDS/VIIRS_Grid_DNB_2d/Data Fields/"

for path in files:
    print(f"\n{path.name}")
    with h5py.File(path, "r") as f:
        rad_ds = f[base + "AllAngle_Composite_Snow_Free"]
        qa_ds = f[base + "AllAngle_Composite_Snow_Free_Quality"]

        print("Radiance attributes:")
        for key in ["_FillValue", "scale_factor", "add_offset", "units", "valid_range"]:
            print(f"  {key}: {rad_ds.attrs.get(key, 'not present')}")

        radiance = rad_ds[:]
        quality = qa_ds[:]

        codes, counts = np.unique(quality, return_counts=True)
        print("Quality flag counts:", dict(zip(codes.tolist(), counts.tolist())))

        good = radiance[np.isfinite(radiance) & (quality == 0) & (radiance > -900)]
        print("Quality-0 radiance percentiles:")
        print(np.percentile(good, [0, 50, 90, 95, 98, 99, 99.9, 100]))
        print("Count above 100:", np.sum(good > 100))

In [ ]:
from pathlib import Path
import h5py
import numpy as np

base = "HDFEOS/GRIDS/VIIRS_Grid_DNB_2d/Data Fields/"
layer = base + "AllAngle_Composite_Snow_Free"
quality_layer = base + "AllAngle_Composite_Snow_Free_Quality"

file_2012 = next(Path("/content").glob("VNP46A4.A2012001.h10v04*.h5"))
file_2023 = next(Path("/content").glob("VNP46A4.A2023001.h10v04*.h5"))

def load_radiance_and_quality(path):
    with h5py.File(path, "r") as f:
        ds = f[layer]
        radiance = ds[:].astype(np.float32)
        quality = f[quality_layer][:]

        fill_value = float(np.asarray(ds.attrs["_FillValue"]).squeeze())

    valid = (
        np.isfinite(radiance)
        & (radiance >= 0)
        & (radiance != fill_value)
        & (quality == 0)
    )
    return radiance, valid

rad_2012, valid_2012 = load_radiance_and_quality(file_2012)
rad_2023, valid_2023 = load_radiance_and_quality(file_2023)

# Compare only pixels that pass the same checks in both years
common_valid = valid_2012 & valid_2023
delta = np.full(rad_2012.shape, np.nan, dtype=np.float32)
delta[common_valid] = rad_2023[common_valid] - rad_2012[common_valid]

# Provisional sound range: cap at the 98th percentile of absolute change
sound_limit = np.percentile(np.abs(delta[common_valid]), 98)
delta_for_sound = np.full(delta.shape, np.nan, dtype=np.float32)
delta_for_sound[common_valid] = np.clip(
    delta[common_valid] / sound_limit, -1, 1
)

print("Pixels valid in both years:", int(common_valid.sum()))
print("Raw change percentiles:", np.percentile(
    delta[common_valid], [1, 50, 99]
))
print("Sound mapping range: -1 to +1")
print("Raw change represented by -1/+1:", round(float(sound_limit), 3))
print("Pixels capped for sound:", int(
    np.sum(np.abs(delta[common_valid]) > sound_limit)
))

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

with h5py.File(file_2012, "r") as f:
    lat = f[base + "lat"][:]
    lon = f[base + "lon"][:]

# Small area around Dubai
rows = np.where((lat >= 40.2) & (lat <= 41.2))[0]
cols = np.where((lon >= -74.6) & (lon <= -73.4))[0]

region = delta[np.ix_(rows, cols)]

origin = "upper" if lat[0] > lat[-1] else "lower"
plt.figure(figsize=(7, 6))
plt.imshow(
    region,
    origin=origin,
    extent=[lon[cols].min(), lon[cols].max(),
            lat[rows].min(), lat[rows].max()],
    cmap="PuOr",
    vmin=-sound_limit,
    vmax=sound_limit,
    interpolation="nearest"
)
plt.colorbar(label="Radiance change (2023 − 2012)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Night-light radiance change near LA")
plt.show()

In [ ]:
import numpy as np
from IPython.display import Audio, display

# Use a scale suited to this Dubai crop
valid_values = region[np.isfinite(region)]
local_limit = np.percentile(np.abs(valid_values), 98)
region_norm = np.clip(region / local_limit, -1, 1)

# Average the crop into 16 × 16 cells
grid_size = 16
height, width = region_norm.shape
grid = np.full((grid_size, grid_size), np.nan, dtype=np.float32)

for row in range(grid_size):
    r0, r1 = row * height // grid_size, (row + 1) * height // grid_size
    for col in range(grid_size):
        c0, c1 = col * width // grid_size, (col + 1) * width // grid_size
        block = region_norm[r0:r1, c0:c1]
        values = block[np.isfinite(block)]
        if values.size:
            grid[row, col] = values.mean()

# Play the grid row by row
sample_rate = 22050
note_seconds = 0.12
fade_seconds = 0.005
audio_parts = []

for value in grid.ravel():
    if not np.isfinite(value):
        audio_parts.append(np.zeros(int(sample_rate * note_seconds)))
        continue

    frequency = 220 + (value + 1) / 2 * 660  # -1 → 220 Hz; +1 → 880 Hz
    amplitude = 0.03 + 0.20 * abs(value)     # Larger changes sound louder

    n = int(sample_rate * note_seconds)
    t = np.arange(n) / sample_rate
    tone = amplitude * np.sin(2 * np.pi * frequency * t)

    fade_n = int(sample_rate * fade_seconds)
    envelope = np.ones(n)
    envelope[:fade_n] = np.linspace(0, 1, fade_n)
    envelope[-fade_n:] = np.linspace(1, 0, fade_n)
    audio_parts.append(tone * envelope)

audio = np.concatenate(audio_parts)

print(f"Crop sound scale: ±{local_limit:.3f} radiance-change units")
print(f"Audio duration: {len(audio) / sample_rate:.1f} seconds")
display(Audio(audio, rate=sample_rate))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

grid_size = grid.shape[0]
sequence = 1
mapping_rows = []

fig, ax = plt.subplots(figsize=(14, 12))
image = ax.imshow(
    grid,
    origin=origin,
    cmap="PuOr",
    vmin=-1,
    vmax=1,
    interpolation="nearest"
)
fig.colorbar(image, ax=ax, label="Normalized radiance change")
ax.set_title("Each grid cell becomes one tone (number = playback order)")
ax.set_xlabel("Column: west to east")
ax.set_ylabel("Grid row")

for row in range(grid_size):
    for col in range(grid_size):
        value = grid[row, col]

        if np.isfinite(value):
            frequency = 220 + (value + 1) / 2 * 660
            amplitude = 0.03 + 0.20 * abs(value)
            label = f"{sequence}\n{value:+.2f}"

            mapping_rows.append({
                "tone_order": sequence,
                "row": row + 1,
                "column": col + 1,
                "normalized_change": round(float(value), 3),
                "pitch_Hz": round(float(frequency), 1),
                "amplitude": round(float(amplitude), 3),
            })
        else:
            frequency = np.nan
            amplitude = 0
            label = f"{sequence}\nno data"

            mapping_rows.append({
                "tone_order": sequence,
                "row": row + 1,
                "column": col + 1,
                "normalized_change": np.nan,
                "pitch_Hz": np.nan,
                "amplitude": 0,
            })

        ax.text(
            col, row, label,
            ha="center", va="center",
            fontsize=6, color="black"
        )
        sequence += 1

ax.set_xticks(range(grid_size))
ax.set_yticks(range(grid_size))
ax.set_xticklabels(range(1, grid_size + 1))
ax.set_yticklabels(range(1, grid_size + 1))
plt.tight_layout()
plt.show()

mapping = pd.DataFrame(mapping_rows)
display(mapping)

In [ ]:
import numpy as np
import wave

from IPython.display import Audio, display

# Match the map's conventional orientation:
# left = west, top = north
grid_geo = grid.copy()

if lon[cols[0]] > lon[cols[-1]]:
    grid_geo = grid_geo[:, ::-1]

if lat[rows[0]] < lat[rows[-1]]:
    grid_geo = grid_geo[::-1, :]

grid_size = grid_geo.shape[0]
sample_rate = 22050
column_seconds = 0.4
samples_per_column = int(sample_rate * column_seconds)

# Top of map = high pitch; bottom = low pitch
# row_frequencies = np.linspace(880, 220, grid_size)
row_frequencies = np.linspace(1320, 330, grid_size)
audio_columns = []

t = np.arange(samples_per_column) / sample_rate
fade_n = int(0.01 * sample_rate)
envelope = np.ones(samples_per_column)
envelope[:fade_n] = np.linspace(0, 1, fade_n)
envelope[-fade_n:] = np.linspace(1, 0, fade_n)

# One chord per column, scanning from west to east
for col in range(grid_size):
    chord = np.zeros(samples_per_column)

    for row in range(grid_size):
        value = grid_geo[row, col]

        if not np.isfinite(value) or value == 0:
            continue

        frequency = row_frequencies[row]
        sine = np.sin(2 * np.pi * frequency * t)

        # Increase = sine tone; decrease = triangle tone
        tone = sine if value > 0 else (2 / np.pi) * np.arcsin(sine)

        # Larger absolute change = louder tone
        chord += tone * (0.18 * abs(value))

    chord = chord / grid_size
    audio_columns.append(chord * envelope)

audio = np.concatenate(audio_columns)

# Prevent clipping while preserving relative loudness
peak = np.max(np.abs(audio))
if peak > 0:
    audio = 0.8 * audio / peak

# Save WAV for the final video
pcm = (audio * 32767).astype(np.int16)
with wave.open("ny_scan.wav", "wb") as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)
    wav.setframerate(sample_rate)
    wav.writeframes(pcm.tobytes())

print(f"Audio duration: {len(audio) / sample_rate:.1f} seconds")
display(Audio(audio, rate=sample_rate))

In [ ]:
import shutil
import subprocess
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(
    grid_geo,
    origin="upper",
    cmap="PuOr",
    vmin=-1,
    vmax=1,
    interpolation="nearest"
)

fig.colorbar(image, ax=ax, label="Normalized radiance change")
ax.set_title("la night-light change: 2012 to 2023")
ax.set_xlabel("West  →  East")
ax.set_ylabel("North  →  South")

# Moving vertical scan bar
bar = ax.axvline(0, color="cyan", linewidth=2.5)
status = ax.text(
    0.02, 0.98, "", transform=ax.transAxes,
    va="top", color="black",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none")
)

fps = 20
frames_per_column = int(fps * column_seconds)
total_frames = grid_size * frames_per_column

def update(frame):
    col = min(frame // frames_per_column, grid_size - 1)
    bar.set_xdata([col, col])
    status.set_text(f"Column {col + 1} of {grid_size}")
    return bar, status

animation = FuncAnimation(
    fig,
    update,
    frames=total_frames,
    interval=1000 / fps,
    blit=False
)

display(HTML(animation.to_jshtml()))

# Export silent video, then combine it with the WAV
if shutil.which("ffmpeg") is None:
    raise RuntimeError("FFmpeg is not available in this Colab runtime.")

animation.save("ny_scan_silent.mp4", writer="ffmpeg", fps=fps, dpi=120)

subprocess.run([
    "ffmpeg", "-y",
    "-i", "ny_scan_silent.mp4",
    "-i", "ny_scan.wav",
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-shortest",
    "ny_scan_with_audio.mp4"
], check=True)

print("Created ny_scan_with_audio.mp4")

In [ ]:
import shutil
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# Make a display-only copy of the crop and orient it to match the audio grid:
# left = west, right = east, top = north, bottom = south
region_display = region.copy()
lon_display = lon[cols].copy()
lat_display = lat[rows].copy()

reverse_cols = lon_display[0] > lon_display[-1]
reverse_rows = lat_display[0] < lat_display[-1]

if reverse_cols:
    region_display = region_display[:, ::-1]
    lon_display = lon_display[::-1]

if reverse_rows:
    region_display = region_display[::-1, :]
    lat_display = lat_display[::-1]

# The audio scans across grid columns
grid_rows, scan_columns = grid_geo.shape

# Find the longitude center of each audio column
width = len(lon_display)
column_centers = []

for col in range(scan_columns):
    c0 = col * width // scan_columns
    c1 = (col + 1) * width // scan_columns
    column_centers.append(np.mean(lon_display[c0:c1]))

# Find the boundaries of the audio columns
column_edges = np.empty(scan_columns + 1)
column_edges[0] = lon_display.min()
column_edges[-1] = lon_display.max()
column_edges[1:-1] = (
    np.asarray(column_centers[:-1]) + np.asarray(column_centers[1:])
) / 2

# Draw the map
display_limit = sound_limit

fig, ax = plt.subplots(figsize=(10, 8))

image = ax.imshow(
    region_display,
    origin="upper",
    extent=[
        lon_display.min(), lon_display.max(),
        lat_display.min(), lat_display.max()
    ],
    cmap="PuOr",
    vmin=-display_limit,
    vmax=display_limit,
    interpolation="nearest",
    aspect="auto"
)

fig.colorbar(image, ax=ax, label="Radiance change (2023 − 2012)")
ax.set_title("Night-light radiance change near Tokyo")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Dark, subtle outline plus a faded grey-white scan bar
bar_outline = ax.axvline(
    column_edges[0],
    color="#16191D",
    linewidth=4,
    alpha=0.40
)
bar = ax.axvline(
    column_edges[0],
    color="#D8DCE0",
    linewidth=2.2,
    alpha=0.75
)

status = ax.text(
    0.02, 0.98, "",
    transform=ax.transAxes,
    va="top",
    color="black",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none")
)

# At 60 fps, the bar advances smoothly while each audio column
# still lasts column_seconds.
fps = 60
frames_per_column = round(fps * column_seconds)
total_frames = scan_columns * frames_per_column

def update(frame):
    # Move continuously from the map's western edge to its eastern edge
    progress = frame / (total_frames - 1)
    scan_position = progress * scan_columns

    x = np.interp(
        scan_position,
        np.arange(scan_columns + 1),
        column_edges
    )

    col = min(int(scan_position), scan_columns - 1)

    bar_outline.set_xdata([x, x])
    bar.set_xdata([x, x])
    status.set_text(f"Column {col + 1} of {scan_columns}")

    return bar_outline, bar, status

animation = FuncAnimation(
    fig,
    update,
    frames=total_frames,
    interval=1000 / fps,
    blit=False
)

display(HTML(animation.to_jshtml(fps=fps)))

# Export the silent animation
if shutil.which("ffmpeg") is None:
    raise RuntimeError("FFmpeg is not available in this Colab runtime.")

animation.save(
    "ny_scan_map_silent.mp4",
    writer="ffmpeg",
    fps=fps,
    dpi=120
)

# Attach the existing audio without changing it
subprocess.run([
    "ffmpeg", "-y",
    "-i", "ny_scan_map_silent.mp4",
    "-i", "ny_scan.wav",
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-shortest",
    "ny_scan_map_with_audio.mp4"
], check=True)

plt.close(fig)
print("Saved: ny_scan_map_with_audio.mp4")